In [0]:
# Databricks notebook source
"""
notebooks/get_active_tables.py

Generates the dynamic list of active table_ids for a given table_group
(DIMENSION or FACT), read straight from the metadata table. The For Each
tasks downstream loop over this list -- adding a new table means adding
a metadata row and dropping in three script files. Nothing about this
job's configuration ever needs to change.
"""

dbutils.widgets.text("catalog", "uc_dev_snt_fdn")
dbutils.widgets.text("table_group", "DIMENSION")

catalog = dbutils.widgets.get("catalog")
table_group = dbutils.widgets.get("table_group")

rows = spark.sql(f"""
    SELECT table_id
    FROM {catalog}.config.ingestion_metadata
    WHERE 
    -- table_group = '{table_group}' AND  Changing to make the table_group is no use now
       table_is_active = true
    ORDER BY table_id
""").collect()

table_ids = [r.table_id for r in rows]

dbutils.jobs.taskValues.set(key="table_ids", value=table_ids)
print(f"Active {table_group} tables picked up for this run: {table_ids}")